# Version 8.0
- Entity to cluster comparison

In [110]:
import faiss
import numpy as np
from sklearn.preprocessing import normalize
from sklearn.feature_extraction.text import TfidfTransformer
from collections import defaultdict
from itertools import combinations
from sentence_transformers import SentenceTransformer, util
from fuzzywuzzy import fuzz
from knowledge_graph_maker import Edge, Node
from generate_graph import generateEdges, createGraph
from transformers import pipeline


## Entity Resolution Class

In [ ]:
class ClusterER:
    def __init__(self, pred2idx=None, sim_threshold=0.65, embedding_model="all-mpnet-base-v2"):
        """
        pred2idx: mapping predicate -> index for relational signatures
        sim_threshold: threshold to merge with existing cluster
        """
        self.model = SentenceTransformer(embedding_model)
        self.ner_pipeline = pipeline(
        "ner",
        model="dslim/bert-base-NER",
        aggregation_strategy="simple"
        )
        self.sim_threshold = sim_threshold

        # FAISS for cluster embeddings
        self.dim = 768  # mpnet embedding size
        self.faiss_index = faiss.IndexFlatIP(self.dim)
        self.int2cluster = {}  # mapping FAISS row -> cluster_id
        self.cluster_counter = 1

        # Clusters
        self.clusters = {}  # cluster_id -> cluster dict

        # Relational signature
        self.pred2idx = pred2idx or {}
        self.num_predicates = len(self.pred2idx)

    # ----------------------
    # Helper: String similarity
    # ----------------------
    def _string_sim(self, a, b):
        return fuzz.token_sort_ratio(a, b) / 100

    # ----------------------
    # Helper: Type compatibility
    # ----------------------
    # def _predict_type(self, entity_text, predicate=None):
    #     # Dummy placeholder: override with NER model if desired
    #     return "UNKNOWN"
    
    def _predict_type(self, entity_text, predicate=None):
        text = f"{entity_text} {predicate}" if predicate else entity_text

        try:
            entities = self.ner_pipeline(text)
        except Exception:
            return "UNKNOWN"

        for ent in entities:
            if ent["word"].lower() in entity_text.lower():
                label = ent["entity_group"]
                return {
                    "PER": "PERSON",
                    "ORG": "ORGANIZATION",
                    "LOC": "LOCATION",
                    "MISC": "MISC"
                }.get(label, "UNKNOWN")

        return "UNKNOWN"

    def _types_compatible(self, type1, type2):
        if type1 == "UNKNOWN" or type2 == "UNKNOWN":
            return True
        return type1 == type2

    # ----------------------
    # FAISS utilities
    # ----------------------
    def _add_to_faiss(self, cluster_id, emb):
        row = self.faiss_index.ntotal
        self.faiss_index.add(emb)
        self.int2cluster[row] = cluster_id

    def _semantic_blocking(self, entity_text, top_k=10):
        """Retrieve candidate clusters via FAISS"""
        if self.faiss_index.ntotal == 0:
            return []

        query_emb = self.model.encode(entity_text)
        query_emb = np.asarray(query_emb, dtype="float32").reshape(1, -1)
        faiss.normalize_L2(query_emb)

        scores, indices = self.faiss_index.search(query_emb, top_k)
        candidates = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1 or score < self.sim_threshold:
                continue
            candidates.append(self.int2cluster[idx])
        return candidates

    # ----------------------
    # Relational signatures
    # ----------------------
    def _compute_candidate_relvec(self, predicate=None, other_entity_id=None):
        """Build candidate relational vector for current triple"""
        vec = np.zeros((1, self.num_predicates * 2), dtype=np.float32)
        if predicate in self.pred2idx:
            p_idx = self.pred2idx[predicate]
            vec[0, p_idx] += 1  # subject role
            if other_entity_id is not None:
                vec[0, p_idx + self.num_predicates] += 1  # object role
        return vec

    def _cluster_relational_similarity(self, cluster_id, candidate_vec=None):
        cluster_vec = self.clusters[cluster_id]["rel_signature"]
        if candidate_vec is None:
            candidate_vec = np.zeros_like(cluster_vec)

        # cosine similarity
        c_norm = cluster_vec / (np.linalg.norm(cluster_vec) + 1e-10)
        e_norm = candidate_vec / (np.linalg.norm(candidate_vec) + 1e-10)
        return float(np.dot(c_norm, e_norm.T))

    # ----------------------
    # Cluster management
    # ----------------------
    def _create_new_cluster(self, entity_text, predicate=None):
        cluster_id = f"C{self.cluster_counter}"
        self.cluster_counter += 1

        emb = self.model.encode(entity_text)
        emb = np.asarray(emb, dtype="float32").reshape(1, -1)
        faiss.normalize_L2(emb)

        rel_vec = self._compute_candidate_relvec(predicate)

        self.clusters[cluster_id] = {
            "members": {entity_text},
            "centroid": emb,
            "type": self._predict_type(entity_text, predicate),
            "neighbors": set(),
            "rel_signature": rel_vec
        }

        self._add_to_faiss(cluster_id, emb)
        return cluster_id, "INSERT"

    def _update_cluster(self, cluster_id, new_entity_text, predicate=None):
        cluster = self.clusters[cluster_id]

        new_emb = self.model.encode(new_entity_text)
        new_emb = np.asarray(new_emb, dtype="float32").reshape(1, -1)
        faiss.normalize_L2(new_emb)

        # Update centroid
        cluster["centroid"] = normalize(cluster["centroid"] + new_emb, norm="l2")
        cluster["members"].add(new_entity_text)

        # Update relational signature
        if predicate:
            candidate_vec = self._compute_candidate_relvec(predicate)
            cluster["rel_signature"] += candidate_vec

        # Rebuild FAISS for simplicity
        self._rebuild_faiss()

    def _rebuild_faiss(self):
        self.faiss_index.reset()
        self.int2cluster = {}
        for cid, c in self.clusters.items():
            self._add_to_faiss(cid, c["centroid"])

    # ----------------------
    # Cluster-level similarity
    # ----------------------
    def _cluster_similarity(self, entity_text, cluster_id, predicate=None, other_entity_id=None):
        cluster = self.clusters[cluster_id]

        # string similarity
        s_sim = max(self._string_sim(entity_text, m) for m in cluster["members"])

        # embedding similarity
        context = f"{entity_text} in relation {predicate}" if predicate else entity_text
        emb_query = self.model.encode(context)
        e_sim = util.cos_sim(emb_query, cluster["centroid"]).item()

        # type similarity
        new_type = self._predict_type(entity_text, predicate)
        p_sim = 1.0 if self._types_compatible(new_type, cluster["type"]) else 0.0

        # relational similarity
        candidate_relvec = self._compute_candidate_relvec(predicate, other_entity_id)
        r_sim = self._cluster_relational_similarity(cluster_id, candidate_relvec)

        # composite
        return 0.25 * s_sim + 0.40 * e_sim + 0.10 * p_sim + 0.25 * r_sim

    # ----------------------
    # Resolve entity
    # ----------------------
    def resolve_entity(self, entity_text, predicate=None, other_entity_id=None):
        candidates = self._semantic_blocking(entity_text)
        best_score = 0
        best_cluster = None

        for cluster_id in candidates:
            score = self._cluster_similarity(entity_text, cluster_id, predicate, other_entity_id)
            if score > best_score:
                best_score = score
                best_cluster = cluster_id

        if best_cluster and best_score >= self.sim_threshold:
            self._update_cluster(best_cluster, entity_text, predicate)
            return best_cluster, "MERGE"

        return self._create_new_cluster(entity_text, predicate)

    # ----------------------
    # Get entity -> cluster mapping
    # ----------------------
    def get_entity_mapping(self):
        mapping = {}
        for cid, cluster in self.clusters.items():
            for m in cluster["members"]:
                mapping[m] = cid
        return mapping

    # ----------------------
    # Evaluation functions
    # ----------------------
    @staticmethod
    def pairwise_links(items):
        if len(items) == 0:
            return set()
        if len(items) == 1:
            return {frozenset(items)}
        return set(frozenset([a, b]) for a, b in combinations(items, 2))

    @staticmethod
    def build_gold_entity_map(gold_clusters):
        gold_entity_map = defaultdict(set)
        for gold_cluster, entities in gold_clusters.items():
            for e in entities:
                gold_entity_map[e].add(gold_cluster)
        return gold_entity_map

    @staticmethod
    def evaluate_cluster_alignment(gold_clusters, predicted_clusters):
        gold_entity_map = ClusterER.build_gold_entity_map(gold_clusters)
        for pred_cluster_id, pred_entities in predicted_clusters.items():
            matched_gold = defaultdict(set)
            for e in pred_entities:
                golds = gold_entity_map.get(e, set())
                if not golds:
                    matched_gold["UNSEEN"].add(e)
                else:
                    for g in golds:
                        matched_gold[g].add(e)
            print("="*50)
            print(f"Predicted Cluster: {pred_cluster_id}")
            print(f"Entities: {pred_entities}")
            if len(matched_gold) == 1:
                print(f"Pure match with gold cluster: {next(iter(matched_gold))}")
            else:
                print("Mixed cluster matches:")
                for g, ents in matched_gold.items():
                    print(f"  - {g}: {ents}")

    def evaluate(self, toy_dataset):
        # build gold clusters
        gold_clusters = defaultdict(set)
        for t in toy_dataset:
            gold_clusters[t["subject_gold"]].add(t["subject"])
            gold_clusters[t["object_gold"]].add(t["object"])

        # predicted clusters
        pred_clusters = defaultdict(set)
        mapping = self.get_entity_mapping()
        for mention, cid in mapping.items():
            pred_clusters[cid].add(mention)

        # pairwise metrics
        gold_links = set()
        for entities in gold_clusters.values():
            gold_links.update(self.pairwise_links(entities))
        pred_links = set()
        for entities in pred_clusters.values():
            pred_links.update(self.pairwise_links(entities))

        tp = len(gold_links & pred_links)
        fp = len(pred_links - gold_links)
        fn = len(gold_links - pred_links)
        precision = tp / (tp + fp + 1e-10)
        recall = tp / (tp + fn + 1e-10)
        f1 = 2 * precision * recall / (precision + recall + 1e-10)

        print("\n=== Evaluation Results ===")
        print(f"Precision: {precision:.4f}")
        print(f"Recall   : {recall:.4f}")
        print(f"F1       : {f1:.4f}")

        print("\n=== Cluster Alignment ===")
        self.evaluate_cluster_alignment(gold_clusters, pred_clusters)
        return precision, recall, f1
    
    def deduplicate_edges(self, list_of_edges, merge_metadata=True):
        """
        Remove or merge duplicate edges after entity resolution.

        Returns:
            unique_edges: list of canonicalized edges
            duplicate_count: number of removed edges
        """

        canonical_edges = {}
        duplicate_count = 0

        for edge in list_of_edges:
            s_text = edge.node_1.name
            o_text = edge.node_2.name
            predicate = edge.relationship

            # Resolve to cluster IDs
            s_cluster = self.get_entity_mapping().get(s_text)
            o_cluster = self.get_entity_mapping().get(o_text)

            # If resolution failed, skip
            if s_cluster is None or o_cluster is None:
                continue

            # Canonical triple key
            key = (s_cluster, predicate, o_cluster)

            if key not in canonical_edges:
                canonical_edges[key] = {
                    "subject_cluster": s_cluster,
                    "predicate": predicate,
                    "object_cluster": o_cluster,
                    "mentions": [(s_text, o_text)],
                    "metadata": [edge.metadata]
                }
            else:
                duplicate_count += 1
                canonical_edges[key]["mentions"].append((s_text, o_text))

                if merge_metadata:
                    canonical_edges[key]["metadata"].append(edge.metadata)

        return list(canonical_edges.values()), duplicate_count
    
    def process_edges(self, list_of_edges):
        for edge in list_of_edges:

            subj = edge.node_1.name
            obj = edge.node_2.name
            pred = edge.relationship

            subj_cluster, _ = self.resolve_entity(subj, predicate=pred)
            obj_cluster, _ = self.resolve_entity(obj, predicate=pred,
                                                other_entity_id=subj_cluster)

        return self.deduplicate_edges_to_graph(list_of_edges)
        
    def deduplicate_edges_to_graph(self, list_of_edges):
        """
        Deduplicate edges but reconstruct them using canonical mention names
        instead of cluster IDs.
        """

        entity_map = self.get_entity_mapping()

        # Step 1: Build cluster -> canonical mention name mapping
        cluster_to_name = {}
        cluster_to_label = {}

        for cluster_id, cluster_data in self.clusters.items():
            # Choose first inserted mention as canonical
            canonical_name = next(iter(cluster_data["members"]))
            cluster_to_name[cluster_id] = canonical_name

        canonical_edges = {}
        new_order = 0

        for edge in list_of_edges:
            s_text = edge.node_1.name
            o_text = edge.node_2.name
            predicate = edge.relationship

            s_cluster = entity_map.get(s_text)
            o_cluster = entity_map.get(o_text)

            if s_cluster is None or o_cluster is None:
                continue

            key = (s_cluster, predicate, o_cluster)

            if key not in canonical_edges:
                canonical_edges[key] = {
                    "subject_cluster": s_cluster,
                    "object_cluster": o_cluster,
                    "predicate": predicate,
                    "subject_label": edge.node_1.label,
                    "object_label": edge.node_2.label,
                    "metadata": [edge.metadata],
                    "order": new_order
                }
                new_order += 1
            else:
                canonical_edges[key]["metadata"].append(edge.metadata)

        # Step 2: Reconstruct graph using canonical mention names
        deduplicated_edges = []

        for data in canonical_edges.values():

            subject_node = Node(
                label=data["subject_label"],
                name=cluster_to_name[data["subject_cluster"]],
                properties={}
            )

            object_node = Node(
                label=data["object_label"],
                name=cluster_to_name[data["object_cluster"]],
                properties={}
            )

            merged_metadata = {
                "merged_from": data["metadata"]
            }

            deduplicated_edges.append(
                Edge(
                    node_1=subject_node,
                    node_2=object_node,
                    relationship=data["predicate"],
                    metadata=merged_metadata,
                    order=data["order"]
                )
            )

        return deduplicated_edges

## Faculty Manual

In [ ]:
# Open the file in read mode
with open('propositions_facultymanual.txt', 'r') as file:
    # Read all lines and store them in a list
    propositions_from_file = [line.strip() for line in file]

print(propositions_from_file)

props = propositions_from_file

list_of_edges = generateEdges(props)

In [ ]:
# 1. Initialize predicate vocab for relational signatures
# pred2idx = {"works_at":0, "owns":1, "located_in":2}
pred2idx = {}

# 2. Initialize resolver
resolver = ClusterER(pred2idx=pred2idx, sim_threshold=0.65)

unique_edges = resolver.process_edges(list_of_edges)

Graph without ER

In [ ]:
if createGraph(list_of_edges):
    print("Success")
else:
    print("Failed")

Graph with ER

In [ ]:
# 1. Initialize predicate vocab for relational signatures
# pred2idx = {"works_at":0, "owns":1, "located_in":2}
pred2idx = {}

# 2. Initialize resolver
resolver = ClusterER(pred2idx=pred2idx, sim_threshold=0.65)

unique_edges = resolver.process_edges(list_of_edges)

In [ ]:
if createGraph(unique_edges):
    print("Success")
else:
    print("Failed")

In [156]:
#20 Questions

faculty_manual_questions = [
    {
        "ID": 1,
        "Question": "What is the official title of this document and its reference number?",
        "Answer": "The official title of this document is \"FACULTY MANUAL\" and its reference number is \"OPR-HRD-D-M-003\"."
    },
    {
        "ID": 2,
        "Question": "When is the date of effectivity for this Faculty Manual and what previous version does it supersede?",
        "Answer": "The date of effectivity for this Faculty Manual is August 1, 2023, and it supersedes the version from December 1, 2019."
    },
    {
        "ID": 3,
        "Question": "What is the purpose of this Faculty Manual, according to the Foreword?",
        "Answer": "According to the Foreword, this Faculty Manual is prepared to provide an overview of National University's policies, rules, regulations, and benefits. It also aims to familiarize faculty with important information and provide employment guidelines to foster a safe and healthy work environment."
    },
    {
        "ID": 4,
        "Question": "What are the \"5 Commandments\" for data privacy compliance emphasized by the Commission on Higher Education (CHED) as mentioned in the Privacy Statement?",
        "Answer": "The \"5 Commandments\" for data privacy compliance emphasized by CHED are: appointing a Data Protection Officer (DPO); conducting a Privacy Impact Assessment; creating a Privacy Management Program; implementing privacy and data protection measures; and performing Breach Reporting Procedure."
    },
    {
        "ID": 5,
        "Question": "Who founded National University and when?",
        "Answer": "National University was founded by Don Mariano Fortunato Jhocson on August 1, 1900."
    },
    {
        "ID": 6,
        "Question": "What is the vision of National University?",
        "Answer": "The vision of National University is to be a dynamic private institution committed to nation-building, recognized internationally in education and research."
    },
    {
        "ID": 7,
        "Question": "What are the types of contracts applicable to employees as listed in Section 5.1 of Chapter 3?",
        "Answer": "The types of contracts applicable to employees include: Probationary Employment Contract (For Full-time Faculty Member), Employment Contract (For regularization), Contractual Full-Time Employment Agreement (Non-Regular faculty), Contractual Employment Contract (For Part-time Faculty Member), Adjunct Faculty Contract, Project-based Employment Contract (For Non-Teaching Personnel), and Consultancy Contract."
    },
    {
        "ID": 8,
        "Question": "What is the policy regarding solicitation and distribution of materials in the campus?",
        "Answer": "For the safety, convenience, and protection of all faculty, National University always prohibits solicitation and distribution of materials in the campus."
    },
    {
        "ID": 9,
        "Question": "What is the maximum score a faculty can achieve in Research and Innovation, and what are some categories for points allocation?",
        "Answer": "A faculty's maximum score in Research and Innovation is 30 points. Categories for points allocation include: Publication: Journal Articles, Publication: Conference Proceedings, Publication: Book Chapters and International Handbook and Encyclopedia, Externally Funded Research/Innovation/Resiliency Project, and Completed Internally Funded Research/Innovation/Resiliency Project."
    },
    {
        "ID": 10,
        "Question": "What is the \"No Work, No Pay\" policy specifically for Part-time Faculty regarding tardiness and absences?",
        "Answer": "There is a \"No Work, No Pay\" policy for Part-time Faculty, and tardiness and absences are computed based on their hourly rate."
    },
    {
        "ID": 11,
        "Question": "What is the current student enrollment for the upcoming academic year across all programs?",
        "Answer": ""
    },
    {
        "ID": 12,
        "Question": "What specific pedagogical approaches are recommended for teaching a master's level course in computer science?",
        "Answer": ""
    },
    {
        "ID": 13,
        "Question": "Who are the current deans of each college, and what are their academic backgrounds?",
        "Answer": ""
    },
    {
        "ID": 14,
        "Question": "What is the total budget allocated for faculty research grants in the next fiscal year?",
        "Answer": ""
    },
    {
        "ID": 15,
        "Question": "What are the technical specifications of the university's learning management system?",
        "Answer": ""
    },
    {
        "ID": 16,
        "Question": "What is the university's strategy for international partnerships in the next five years?",
        "Answer": ""
    },
    {
        "ID": 17,
        "Question": "How many faculty members are expected to retire in the next three years?",
        "Answer": ""
    },
    {
        "ID": 18,
        "Question": "What is the average student-to-faculty ratio in undergraduate programs?",
        "Answer": ""
    },
    {
        "ID": 19,
        "Question": "What are the detailed architectural plans for the new campus building under construction?",
        "Answer": ""
    },
    {
        "ID": 20,
        "Question": "What is the university's market share in tertiary education in the Philippines compared to other private institutions?",
        "Answer": ""
    }
]

In [157]:
from query_graph import QueryGraph
from langchain_ollama import ChatOllama

# model = "llama3-chatqa:8b"
model = "gemma3:12b-it-qat"

ollama_llm = ChatOllama(
    model = model,
    temperature = 0.8,
    num_predict = 256,
)

qg = QueryGraph(lm=ollama_llm)

In [158]:
import json

answers = []

for i in faculty_manual_questions:
    
    answerdict = {
        "ID": i['ID'],
        "Question": i['Question'],
        "GroundTruth": i['Answer'],
        "Prediction": ""
    }
    
    print(i['Question'])
    print(i['Answer'])
    
    question = i['Question']
    answer = i['Answer']

    req = qg.get_requirements(question)
    result = qg.answer_question(question, req.content)
    
    if result is not None:
        answerdict["Prediction"] = result['result']
    
    answers.append(answerdict)

# Save to JSON file
with open("results_gemma12b_withoutER_facultymanual.json", "w", encoding="utf-8") as f:
    json.dump(answers, f, indent=4, ensure_ascii=False)

print("Saved to answers.json")

What is the official title of this document and its reference number?
The official title of this document is "FACULTY MANUAL" and its reference number is "OPR-HRD-D-M-003".


> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (w)-[r1]-(x)
WHERE (
    toLower(r1.metadata) =~ '^.*\\b(document)\\w*\\b.*$' OR
    toLower(r1.description) =~ '^.*\\b(document)\\w*\\b.*$'
)
RETURN DISTINCT x.title, x.referenceNumber

Failed to read from defunct connection IPv4Address(('localhost', 7687)) (ResolvedIPv4Address(('127.0.0.1', 7687)))
When is the date of effectivity for this Faculty Manual and what previous version does it supersede?
The date of effectivity for this Faculty Manual is August 1, 2023, and it supersedes the version from December 1, 2019.


> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (w)-[r1]-(x)
WHERE (
    toLower(w.metadata) =~ '^.*\\b(faculty manual)\\w*\\b.*$' OR
    toLower(w.description) =~ '^.*\\b(faculty manual)\\w*\\b.*$'
)

### EM and F1

In [160]:
answers = [
  {
    "ID": 1,
    "Question": "What is the official title of this document and its reference number?",
    "GroundTruth": "The official title of this document is \"FACULTY MANUAL\" and its reference number is \"OPR-HRD-D-M-003\".",
    "Prediction": ""
  },
  {
    "ID": 2,
    "Question": "When is the date of effectivity for this Faculty Manual and what previous version does it supersede?",
    "GroundTruth": "The date of effectivity for this Faculty Manual is August 1, 2023, and it supersedes the version from December 1, 2019.",
    "Prediction": ""
  },
  {
    "ID": 3,
    "Question": "What is the purpose of this Faculty Manual, according to the Foreword?",
    "GroundTruth": "According to the Foreword, this Faculty Manual is prepared to provide an overview of National University's policies, rules, regulations, and benefits. It also aims to familiarize faculty with important information and provide employment guidelines to foster a safe and healthy work environment.",
    "Prediction": ""
  },
  {
    "ID": 4,
    "Question": "What are the \"5 Commandments\" for data privacy compliance emphasized by the Commission on Higher Education (CHED) as mentioned in the Privacy Statement?",
    "GroundTruth": "The \"5 Commandments\" for data privacy compliance emphasized by CHED are: appointing a Data Protection Officer (DPO); conducting a Privacy Impact Assessment; creating a Privacy Management Program; implementing privacy and data protection measures; and performing Breach Reporting Procedure.",
    "Prediction": ""
  },
  {
    "ID": 5,
    "Question": "Who founded National University and when?",
    "GroundTruth": "National University was founded by Don Mariano Fortunato Jhocson on August 1, 1900.",
    "Prediction": ""
  },
  {
    "ID": 6,
    "Question": "What is the vision of National University?",
    "GroundTruth": "The vision of National University is to be a dynamic private institution committed to nation-building, recognized internationally in education and research.",
    "Prediction": "National University Vision, Mission, Dynamic Filipinism and Goals.\n"
  },
  {
    "ID": 7,
    "Question": "What are the types of contracts applicable to employees as listed in Section 5.1 of Chapter 3?",
    "GroundTruth": "The types of contracts applicable to employees include: Probationary Employment Contract (For Full-time Faculty Member), Employment Contract (For regularization), Contractual Full-Time Employment Agreement (Non-Regular faculty), Contractual Employment Contract (For Part-time Faculty Member), Adjunct Faculty Contract, Project-based Employment Contract (For Non-Teaching Personnel), and Consultancy Contract.",
    "Prediction": ""
  },
  {
    "ID": 8,
    "Question": "What is the policy regarding solicitation and distribution of materials in the campus?",
    "GroundTruth": "For the safety, convenience, and protection of all faculty, National University always prohibits solicitation and distribution of materials in the campus.",
    "Prediction": "National University prohibits solicitation and distribution of materials on the campus.\n"
  },
  {
    "ID": 9,
    "Question": "What is the maximum score a faculty can achieve in Research and Innovation, and what are some categories for points allocation?",
    "GroundTruth": "A faculty's maximum score in Research and Innovation is 30 points. Categories for points allocation include: Publication: Journal Articles, Publication: Conference Proceedings, Publication: Book Chapters and International Handbook and Encyclopedia, Externally Funded Research/Innovation/Resiliency Project, and Completed Internally Funded Research/Innovation/Resiliency Project.",
    "Prediction": ""
  },
  {
    "ID": 10,
    "Question": "What is the \"No Work, No Pay\" policy specifically for Part-time Faculty regarding tardiness and absences?",
    "GroundTruth": "There is a \"No Work, No Pay\" policy for Part-time Faculty, and tardiness and absences are computed based on their hourly rate.",
    "Prediction": ""
  },
  {
    "ID": 11,
    "Question": "What is the current student enrollment for the upcoming academic year across all programs?",
    "GroundTruth": "",
    "Prediction": ""
  },
  {
    "ID": 12,
    "Question": "What specific pedagogical approaches are recommended for teaching a master's level course in computer science?",
    "GroundTruth": "",
    "Prediction": ""
  },
  {
    "ID": 13,
    "Question": "Who are the current deans of each college, and what are their academic backgrounds?",
    "GroundTruth": "",
    "Prediction": "National University Hymn, official activities, College/School Deans, Colleges/Schools, Colleges/Schools, Teaching Personnel, Dean, new hire, College Dean, Academic Director, Campus, program competencies, COMEX Coordinator, Faculty, Proper greetings, recipient, community service or institutional, college, or professional committee, COMEX.\nTheir academic backgrounds are not provided.\n"
  },
  {
    "ID": 14,
    "Question": "What is the total budget allocated for faculty research grants in the next fiscal year?",
    "GroundTruth": "",
    "Prediction": ""
  },
  {
    "ID": 15,
    "Question": "What are the technical specifications of the university's learning management system?",
    "GroundTruth": "",
    "Prediction": ""
  },
  {
    "ID": 16,
    "Question": "What is the university's strategy for international partnerships in the next five years?",
    "GroundTruth": "",
    "Prediction": ""
  },
  {
    "ID": 17,
    "Question": "How many faculty members are expected to retire in the next three years?",
    "GroundTruth": "",
    "Prediction": "0\n"
  },
  {
    "ID": 18,
    "Question": "What is the average student-to-faculty ratio in undergraduate programs?",
    "GroundTruth": "",
    "Prediction": ""
  },
  {
    "ID": 19,
    "Question": "What are the detailed architectural plans for the new campus building under construction?",
    "GroundTruth": "",
    "Prediction": ""
  },
  {
    "ID": 20,
    "Question": "What is the university's market share in tertiary education in the Philippines compared to other private institutions?",
    "GroundTruth": "",
    "Prediction": ""
  }
]


In [161]:
import re
import string
from rouge_score import rouge_scorer

def normalize_answer(s):
    """Lower text and remove punctuation, articles and extra whitespace."""

    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punctuation(text):
        return text.translate(str.maketrans('', '', string.punctuation))

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punctuation(lower(s))))

def exact_match_score(prediction, ground_truth):
    return normalize_answer(prediction) == normalize_answer(ground_truth)

In [162]:
resultlist = []
predictions = []
ground_truths = []

for i in faculty_manual_questions:
    for j in answers:
        if i['ID'] == j['ID']:
            answerdict = {
                "Question": i['Question'],  
                "GroundTruth": i['Answer'],
                "Prediction": j['Prediction']
            }
    resultlist.append(answerdict)

# Compute EM score per QA pair
em_scores = [exact_match_score(item["Prediction"], item["GroundTruth"]) for item in resultlist]

# Compute overall EM accuracy
em_accuracy = sum(em_scores) / len(em_scores)

print(f"Exact Match Accuracy: {em_accuracy:.2%}")

for i in resultlist:
    ground_truths.append(i['GroundTruth'])
    predictions.append(i['Prediction'])

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

total_f1 = 0

for gt, pred in zip(ground_truths, predictions):
    score = scorer.score(gt, pred)
    total_f1 += score['rougeL'].fmeasure

average_f1 = total_f1 / len(ground_truths)

print("Average ROUGE-L F1:", average_f1)

Exact Match Accuracy: 40.00%
Average ROUGE-L F1: 0.04458333333333333


## Research Manual

In [ ]:
def clean_edges_for_neo4j(edges):
    """
    Take a list of Edge objects and return a cleaned version suitable for Neo4j insertion.
    - Replaces spaces in labels with underscores.
    - Escapes quotes in names.
    - Keeps metadata intact but ensures all values are strings and quotes are escaped.
    Returns a new list of Edge objects.
    """
    from copy import deepcopy
    cleaned_edges = []

    for edge in edges:
        # Deepcopy to avoid modifying original objects
        new_edge = deepcopy(edge)

        # Clean node labels (replace spaces with underscores)
        new_edge.node_1.label = new_edge.node_1.label.replace(" ", "_")
        new_edge.node_2.label = new_edge.node_2.label.replace(" ", "_")

        # Clean node names (escape double quotes)
        new_edge.node_1.name = new_edge.node_1.name.replace('"', '\\"')
        new_edge.node_2.name = new_edge.node_2.name.replace('"', '\\"')

        # Convert all metadata values to strings and escape quotes
        new_metadata = {}
        for k, v in new_edge.metadata.items():
            new_metadata[k] = str(v).replace('"', '\\"')
        new_edge.metadata = new_metadata

        # Node properties: ensure all values are strings and escape quotes
        for node in [new_edge.node_1, new_edge.node_2]:
            new_props = {}
            for k, v in node.properties.items():
                new_props[k] = str(v).replace('"', '\\"')
            node.properties = new_props

        cleaned_edges.append(new_edge)

    return cleaned_edges

In [ ]:
# Open the file in read mode
with open('propositions_researchmanual.txt', 'r') as file:
    # Read all lines and store them in a list
    propositions_from_file = [line.strip() for line in file]

print(propositions_from_file)

props = propositions_from_file

list_of_edges = generateEdges(props)

With ER

In [ ]:
list_of_edges

In [ ]:
# 1. Initialize predicate vocab for relational signatures
# pred2idx = {"works_at":0, "owns":1, "located_in":2}
pred2idx = {}

# 2. Initialize resolver
resolver = ClusterER(pred2idx=pred2idx, sim_threshold=0.65)

unique_edges = resolver.process_edges(list_of_edges)

In [ ]:
len(unique_edges)

In [ ]:
clean_unique_edges = clean_edges_for_neo4j(unique_edges)

In [ ]:
len(clean_unique_edges)

In [ ]:
if createGraph(clean_unique_edges):
    print("Success")
else:
    print("Failed")

Without ER

In [ ]:
clean_edges = clean_edges_for_neo4j(list_of_edges)

In [ ]:
clean_edges

In [ ]:
if createGraph(clean_edges):
    print("Success")
else:
    print("Failed")

In [138]:
research_manual_questions = [
    {
        "ID": 1,
        "Question": "What is the primary purpose of the National University Research Manual?",
        "Answer": "The Manual provides an overview of National University's policies, processes, and regulations for the conduct of research, aiming to familiarize constituents with important information about Research and Innovation, and provide guidelines for efficient and ethical research within NU Manila and other NU campuses."
    },
    {
        "ID": 2,
        "Question": "What is the reference number of this Research Manual?",
        "Answer": "The reference number of the Research Manual is RAD-RS-D-M-001."
    },
    {
        "ID": 3,
        "Question": "What are the three university-wide research centers established by National University?",
        "Answer": "The three university-wide research centers are the Center for Research, the Center for Innovation and Entrepreneurship, and the Center for Resilient Philippines."
    },
    {
        "ID": 4,
        "Question": "What is the vision of the Center for Entrepreneurship?",
        "Answer": "The Center for Entrepreneurship is envisioned to be an inclusive, realistic, and collaborative community."
    },
    {
        "ID": 5,
        "Question": "What are some of the core competencies identified by the Center for Resilient Philippines (CRP)?",
        "Answer": "The core competencies of the CRP include disaster resilience from social/political, economic, and physical sciences perspectives; new technology-enabled mechanisms for resilient community planning and design; development of innovative national and local resilience policies; private sector engagement in disaster resilience; capacity building for disaster mitigation and reconstruction; networking with the academic community; and community engagement and participation in reconstruction."
    },
    {
        "ID": 6,
        "Question": "Name at least three research goals of the National University Research Agenda.",
        "Answer": "Some research goals include: to recruit, develop, and retain faculty researchers; to provide proactive service to the academic community on research-related endeavors; to assist faculty researchers in publishing outputs in top international journals; to obtain and manage externally funded research projects; and to foster collaboration with local and international higher education institutions."
    },
    {
        "ID": 7,
        "Question": "What are some of the Research Themes outlined in the National University Research Agenda?",
        "Answer": "Research Themes include Food, Nutrition, and Health; Emerging Industries on the Fourth Industrial Revolution; Development of vaccines and diagnostic kits using indigenous materials; Disaster risk management; Tourism and Pollution control; Climate change; Sports Technology; Education and learning innovations; and Business sophistication."
    },
    {
        "ID": 8,
        "Question": "Which colleges or campuses have their research agenda defined in the manual?",
        "Answer": "The manual defines the research agendas for the College of Education, Arts and Sciences (CEAS), College of Business and Accountancy (CBA), College of Architecture (COA), College of Engineering (COE), College of Computing and Information Technologies (CCIT), College of Allied Health (CAH), College of Dentistry (COD), and College of Tourism and Hospitality Management (CTHM)."
    },
    {
        "ID": 9,
        "Question": "What is the Journal of Sciences, Technology, and Arts (JSTAR)?",
        "Answer": "JSTAR is the official, peer-reviewed, open-access publication of National University (Philippines), published annually by the Center for Research, providing a venue for students, faculty, non-teaching personnel, and industry practitioners to share research works, empirical studies, and theories related to science, technology, and arts."
    },
    {
        "ID": 10,
        "Question": "According to the Authorship Policy, what is the minimum total points an individual needs in the co-authorship scoring system to share authorship?",
        "Answer": "Anyone achieving a total of 25 points in the co-authorship scoring system shares authorship."
    },
    {
        "ID": 11,
        "Question": "What is the current budget allocated for the Center for Innovation and Entrepreneurship for the upcoming fiscal year?",
        "Answer": ""
    },
    {
        "ID": 12,
        "Question": "Who is the specific contact person for submitting research proposals for the College of Allied Health, including their email address and phone number?",
        "Answer": ""
    },
    {
        "ID": 13,
        "Question": "What are the detailed procedures and forms required for intellectual property registration for a patent developed by a faculty member?",
        "Answer": ""
    },
    {
        "ID": 14,
        "Question": "How many research projects were successfully published in international journals last year, broken down by college?",
        "Answer": ""
    },
    {
        "ID": 15,
        "Question": "What is the average duration of a research project from proposal submission to final dissemination of results?",
        "Answer": ""
    },
    {
        "ID": 16,
        "Question": "Are there any specific grants or funding opportunities available exclusively for student-led research initiatives?",
        "Answer": ""
    },
    {
        "ID": 17,
        "Question": "What is the protocol for handling research misconduct cases involving external collaborators?",
        "Answer": ""
    },
    {
        "ID": 18,
        "Question": "Which specific software or tools are recommended by the Office of the Vice President for Research and Development for data analysis in research projects?",
        "Answer": ""
    },
    {
        "ID": 19,
        "Question": "What are the benefits or incentives offered to faculty members who successfully obtain external funding for their research?",
        "Answer": ""
    },
    {
        "ID": 20,
        "Question": "When is the next scheduled workshop or training session for new faculty members on research ethics and compliance?",
        "Answer": ""
    }
]

In [139]:
from query_graph import QueryGraph

# model = "llama3-chatqa:8b"
model = "gemma3:12b-it-qat"

ollama_llm = ChatOllama(
    model = model,
    temperature = 0.8,
    num_predict = 256,
)

qg = QueryGraph(lm=ollama_llm)

In [140]:
import json

answers = []

for i in research_manual_questions:
    
    answerdict = {
        "ID": i['ID'],
        "Question": i['Question'],
        "GroundTruth": i['Answer'],
        "Prediction": ""
    }
    
    print(i['Question'])
    print(i['Answer'])
    
    question = i['Question']
    answer = i['Answer']

    req = qg.get_requirements(question)
    result = qg.answer_question(question, req.content)
    
    if result is not None:
        answerdict["Prediction"] = result['result']
    
    answers.append(answerdict)

# Save to JSON file
with open("results_gemma12b_withoutER_researchmanual.json", "w", encoding="utf-8") as f:
    json.dump(answers, f, indent=4, ensure_ascii=False)

print("Saved to answers.json")

What is the primary purpose of the National University Research Manual?
The Manual provides an overview of National University's policies, processes, and regulations for the conduct of research, aiming to familiarize constituents with important information about Research and Innovation, and provide guidelines for efficient and ethical research within NU Manila and other NU campuses.


> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (w)-[r1]-(x)
WHERE (
    toLower(r1.metadata) =~ '^.*\\b(national university research manual)\\w*\\b.*$' OR
    toLower(r1.description) =~ '^.*\\b(national university research manual)\\w*\\b.*$'
)
RETURN DISTINCT r1.metadata, r1.description

Failed to write data to connection IPv4Address(('localhost', 7687)) (ResolvedIPv4Address(('127.0.0.1', 7687)))
What is the reference number of this Research Manual?
The reference number of the Research Manual is RAD-RS-D-M-001.


> Entering new GraphCypherQAChain chain...
Generated Cypher:
cyphe

In [142]:
answers = [
  {
    "ID": 1,
    "Question": "What is the primary purpose of the National University Research Manual?",
    "GroundTruth": "The Manual provides an overview of National University's policies, processes, and regulations for the conduct of research, aiming to familiarize constituents with important information about Research and Innovation, and provide guidelines for efficient and ethical research within NU Manila and other NU campuses.",
    "Prediction": ""
  },
  {
    "ID": 2,
    "Question": "What is the reference number of this Research Manual?",
    "GroundTruth": "The reference number of the Research Manual is RAD-RS-D-M-001.",
    "Prediction": ""
  },
  {
    "ID": 3,
    "Question": "What are the three university-wide research centers established by National University?",
    "GroundTruth": "The three university-wide research centers are the Center for Research, the Center for Innovation and Entrepreneurship, and the Center for Resilient Philippines.",
    "Prediction": "university-wide research centers\n"
  },
  {
    "ID": 4,
    "Question": "What is the vision of the Center for Entrepreneurship?",
    "GroundTruth": "The Center for Entrepreneurship is envisioned to be an inclusive, realistic, and collaborative community.",
    "Prediction": "To be an inclusive, realistic, and collaborative community.\n"
  },
  {
    "ID": 5,
    "Question": "What are some of the core competencies identified by the Center for Resilient Philippines (CRP)?",
    "GroundTruth": "The core competencies of the CRP include disaster resilience from social/political, economic, and physical sciences perspectives; new technology-enabled mechanisms for resilient community planning and design; development of innovative national and local resilience policies; private sector engagement in disaster resilience; capacity building for disaster mitigation and reconstruction; networking with the academic community; and community engagement and participation in reconstruction.",
    "Prediction": ""
  },
  {
    "ID": 6,
    "Question": "Name at least three research goals of the National University Research Agenda.",
    "GroundTruth": "Some research goals include: to recruit, develop, and retain faculty researchers; to provide proactive service to the academic community on research-related endeavors; to assist faculty researchers in publishing outputs in top international journals; to obtain and manage externally funded research projects; and to foster collaboration with local and international higher education institutions.",
    "Prediction": ""
  },
  {
    "ID": 7,
    "Question": "What are some of the Research Themes outlined in the National University Research Agenda?",
    "GroundTruth": "Research Themes include Food, Nutrition, and Health; Emerging Industries on the Fourth Industrial Revolution; Development of vaccines and diagnostic kits using indigenous materials; Disaster risk management; Tourism and Pollution control; Climate change; Sports Technology; Education and learning innovations; and Business sophistication.",
    "Prediction": "I don't know.\n"
  },
  {
    "ID": 8,
    "Question": "Which colleges or campuses have their research agenda defined in the manual?",
    "GroundTruth": "The manual defines the research agendas for the College of Education, Arts and Sciences (CEAS), College of Business and Accountancy (CBA), College of Architecture (COA), College of Engineering (COE), College of Computing and Information Technologies (CCIT), College of Allied Health (CAH), College of Dentistry (COD), and College of Tourism and Hospitality Management (CTHM).",
    "Prediction": ""
  },
  {
    "ID": 9,
    "Question": "What is the Journal of Sciences, Technology, and Arts (JSTAR)?",
    "GroundTruth": "JSTAR is the official, peer-reviewed, open-access publication of National University (Philippines), published annually by the Center for Research, providing a venue for students, faculty, non-teaching personnel, and industry practitioners to share research works, empirical studies, and theories related to science, technology, and arts.",
    "Prediction": ""
  },
  {
    "ID": 10,
    "Question": "According to the Authorship Policy, what is the minimum total points an individual needs in the co-authorship scoring system to share authorship?",
    "GroundTruth": "Anyone achieving a total of 25 points in the co-authorship scoring system shares authorship.",
    "Prediction": ""
  },
  {
    "ID": 11,
    "Question": "What is the current budget allocated for the Center for Innovation and Entrepreneurship for the upcoming fiscal year?",
    "GroundTruth": "",
    "Prediction": ""
  },
  {
    "ID": 12,
    "Question": "Who is the specific contact person for submitting research proposals for the College of Allied Health, including their email address and phone number?",
    "GroundTruth": "",
    "Prediction": ""
  },
  {
    "ID": 13,
    "Question": "What are the detailed procedures and forms required for intellectual property registration for a patent developed by a faculty member?",
    "GroundTruth": "",
    "Prediction": ""
  },
  {
    "ID": 14,
    "Question": "How many research projects were successfully published in international journals last year, broken down by college?",
    "GroundTruth": "",
    "Prediction": ""
  },
  {
    "ID": 15,
    "Question": "What is the average duration of a research project from proposal submission to final dissemination of results?",
    "GroundTruth": "",
    "Prediction": ""
  },
  {
    "ID": 16,
    "Question": "Are there any specific grants or funding opportunities available exclusively for student-led research initiatives?",
    "GroundTruth": "",
    "Prediction": ""
  },
  {
    "ID": 17,
    "Question": "What is the protocol for handling research misconduct cases involving external collaborators?",
    "GroundTruth": "",
    "Prediction": ""
  },
  {
    "ID": 18,
    "Question": "Which specific software or tools are recommended by the Office of the Vice President for Research and Development for data analysis in research projects?",
    "GroundTruth": "",
    "Prediction": ""
  },
  {
    "ID": 19,
    "Question": "What are the benefits or incentives offered to faculty members who successfully obtain external funding for their research?",
    "GroundTruth": "",
    "Prediction": ""
  },
  {
    "ID": 20,
    "Question": "When is the next scheduled workshop or training session for new faculty members on research ethics and compliance?",
    "GroundTruth": "",
    "Prediction": ""
  }
]


### EM and F1

In [143]:
import re
import string
from rouge_score import rouge_scorer

def normalize_answer(s):
    """Lower text and remove punctuation, articles and extra whitespace."""

    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punctuation(text):
        return text.translate(str.maketrans('', '', string.punctuation))

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punctuation(lower(s))))

def exact_match_score(prediction, ground_truth):
    return normalize_answer(prediction) == normalize_answer(ground_truth)

In [144]:
resultlist = []
predictions = []
ground_truths = []

for i in research_manual_questions:
    for j in answers:
        if i['ID'] == j['ID']:
            answerdict = {
                "Question": i['Question'],  
                "GroundTruth": i['Answer'],
                "Prediction": j['Prediction']
            }
    resultlist.append(answerdict)

# Compute EM score per QA pair
em_scores = [exact_match_score(item["Prediction"], item["GroundTruth"]) for item in resultlist]

# Compute overall EM accuracy
em_accuracy = sum(em_scores) / len(em_scores)

print(f"Exact Match Accuracy: {em_accuracy:.2%}")

for i in resultlist:
    ground_truths.append(i['GroundTruth'])
    predictions.append(i['Prediction'])

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

total_f1 = 0

for gt, pred in zip(ground_truths, predictions):
    score = scorer.score(gt, pred)
    total_f1 += score['rougeL'].fmeasure

average_f1 = total_f1 / len(ground_truths)

print("Average ROUGE-L F1:", average_f1)

Exact Match Accuracy: 50.00%
Average ROUGE-L F1: 0.05117845117845118
